# 🕵️ Data Designer Tutorial: Fraud Detection Conversations

#### 📚 What you'll learn

This notebook demonstrates the basics of Data Designer by generating a "Fraud Detection Conversation" dataset.

Example interactions:
- Customer: "Why was my credit card declined?"
- Agent: "It looks like there were multiple overseas transactions."
- Customer: "I didn't make those purchases."
- Agent: "We'll block the card immediately."

Data Designer can generate:
- Different ages
- Different countries
- Different amounts
- Different merchants
- Different languages
- Different scam methods (ATM Cash Withdrawal, POS Purchase, Apple Pay, Google Pay, Wire Transfer, Crypto Exchange, Gift Card Scam, Romance Scam, Investment Scam, SIM Swap)

This ultimately yields thousands of fraud conversations.

This can be directly trained for:

```
LLM, Fraud Classifier, Call Summary, and Risk Detection.
```

このノートブックでは、「不正検出会話」データセットを生成することで、Data Designerの基本操作を説明します。

会話例：
- 顧客：「なぜクレジットカードが拒否されたのですか？」

- エージェント：「海外での取引が複数件発生しているようです。」

- 顧客：「私はそのような購入はしていません。」

- エージェント：「すぐにカードを停止します。」

Data Designerでは、以下のデータを生成できます。
- 年齢層別
- 国籍別
- 金額別
- 加盟店別
- 言語別
- 詐欺の手口別（ATM現金引き出し、POS決済、Apple Pay、Google Pay、電信送金、仮想通貨交換、ギフトカード詐欺、ロマンス詐欺、投資詐欺、SIMスワップ）

これにより、最終的に数千件の不正検出会話データが生成されます。

このデータは、以下の用途に直接学習させることができます。

```
LLM（論理レベルモデル）、不正分類器、通話サマリー、リスク検出

```

### 📦 Import Data Designer

- `data_designer.config` provides access to the configuration API.
- `DataDesigner` is the main interface for data generation.

日本語: `data_designer.config` は設定APIへのアクセスを提供し、`DataDesigner` はデータ生成の主要インターフェースです。

In [18]:
# ! export NVIDIA_API_KEY="" # TODO copy your nvidia-api-key here:
# https://build.nvidia.com/settings/api-keys
# login with your email box (or register)
# click "Generate API key"
import os

os.environ["NVIDIA_API_KEY"] = "nvapi-***" # TODO replace with your true api key!!!
print(os.getenv("NVIDIA_API_KEY"))

nvapi-***


In [2]:
# ! pip install data-designer

In [3]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ⚙️ Initialize the Data Designer interface

- `DataDesigner` is the main object responsible for managing the data generation process.
- When initialized without arguments, the default model providers are used.

日本語: `DataDesigner` はデータ生成プロセスを管理する主要オブジェクトです。引数なしで初期化するとデフォルトのモデルプロバイダが使用されます。

In [4]:
data_designer = DataDesigner()

### 🎛️ Define model configurations

- Each `ModelConfig` defines a model that can be used during generation.
- The "model alias" is used to reference the model in the Data Designer config.
- The "model provider" is the external service that hosts the model (default: build.nvidia.com).

日本語: 各 `ModelConfig` は生成時に使用できるモデルを定義します。「モデルエイリアス」は設定内でモデルを参照するために使用され、「モデルプロバイダ」はモデルをホストする外部サービスです（デフォルトは build.nvidia.com）。

In [5]:
MODEL_PROVIDER = "nvidia"
MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b"
MODEL_ALIAS = "nemotron-nano-v3"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=0.8,
            top_p=0.7,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        ),
    )
]

### 🏗️ Initialize the Data Designer Config Builder

- The config builder provides an intuitive interface for building the dataset schema and generation process.
- The list of model configs is provided at initialization.

日本語: 設定ビルダーはデータセットスキーマと生成プロセスを構築するための直感的なインターフェースを提供します。モデル設定のリストは初期化時に渡します。

In [6]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

## 🎲 Sampler columns: demographic and fraud context

We'll define sampler columns for:
- Customer age
- Country (where the transaction originates)
- Transaction amount (USD)
- Merchant category (e.g., Retail, Travel, Entertainment, etc.)
- Language (for conversation output)
- Scam method (the type of fraud)
- Fraud indicator (whether the transaction is fraudulent or not)

These samplers will drive diversity in the generated conversations.

日本語: サンプラー列を定義して、年齢、国、取引金額、マーチャントカテゴリ、言語、詐欺手法、詐欺フラグなどの多様性を確保します。

In [7]:
# Customer age (18-80)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="age",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=18, high=80),
        convert_to="int",
    )
)

# Country (ISO country codes or names)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="country",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "United States", "United Kingdom", "Canada", "Australia", "Germany",
                "France", "Spain", "Italy", "Japan", "Brazil", "Mexico", "India",
                "South Africa", "Nigeria", "Singapore", "UAE", "Netherlands", "Sweden"
            ]
        ),
    )
)

# Transaction amount (USD)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="amount_usd",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=10, high=10000),
        convert_to="int",
    )
)

# Merchant category
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="merchant_category",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Retail Store", "Online Shopping", "Travel & Hospitality",
                "Entertainment", "Food & Dining", "Gas Station", "Healthcare",
                "Education", "Financial Services", "Telecom", "Utility"
            ]
        ),
    )
)

# Language for the conversation (we'll generate in multiple languages)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="language",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Japanese"] #["English", "Spanish", "French", "German", "Japanese", "Portuguese", "Hindi"]
        ),
    )
)

# Scam method (type of fraud)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="scam_method",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "ATM Cash Withdrawal", "POS Purchase", "Apple Pay", "Google Pay",
                "Wire Transfer", "Crypto Exchange", "Gift Card Scam",
                "Romance Scam", "Investment Scam", "SIM Swap"
            ]
        ),
    )
)

# Fraud indicator (1 = fraudulent, 0 = legitimate, to get some legitimate conversations as well)
# We'll set weight to 0.8 for fraud, 0.2 for legitimate to have mostly fraud cases but some normal ones.
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="is_fraud",
        sampler_type=dd.SamplerType.BERNOULLI,
        params=dd.BernoulliSamplerParams(p=0.8),
        convert_to="int",
    )
)

# Optionally validate
data_designer.validate(config_builder)

[22:15:01] [INFO] ✅ Validation passed


## 🦜 LLM-generated conversation columns

We'll generate two columns:
- `customer_message`: the customer's initial complaint or question about the transaction.
- `agent_response`: the bank/agent's reply, including fraud detection actions.

We use Jinja templating to reference the sampler columns (age, country, amount, merchant, language, scam_method, is_fraud).

If `is_fraud` = 1, the conversation will be about a fraudulent transaction; if 0, it will be a legitimate inquiry.

日本語: LLMを用いて顧客のメッセージとエージェントの返答を生成します。Jinjaテンプレートでサンプラー列を参照し、詐欺か否かで内容を変えます。

In [8]:
# Customer message
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="customer_message",
        prompt=(
            "You are a customer aged {{ age }} from {{ country }}. "
            "You recently had a transaction of ${{ amount_usd }} at a {{ merchant_category }}. "
            "The transaction method is {{ scam_method }}. "
            "{% if is_fraud == 1 %}"
            "This transaction was fraudulent and you did not authorize it. "
            "Write a short message (1-2 sentences) to the bank's fraud detection team, expressing your concern and asking for help. "
            "{% else %}"
            "This transaction is legitimate and you recognize it. "
            "Write a short message (1-2 sentences) to the bank's fraud detection team confirming this is your transaction and asking if everything is okay. "
            "{% endif %}"
            "The message should be in {{ language }}. "
            "Do not add any meta-commentary; only output the customer's message."
        ),
        model_alias=MODEL_ALIAS,
    )
)

# Agent response
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="agent_response",
        prompt=(
            "You are a bank fraud detection agent. A customer (age {{ age }}, from {{ country }}) reported "
            "a transaction of ${{ amount_usd }} at a {{ merchant_category }} using {{ scam_method }}. "
            "{% if is_fraud == 1 %}"
            "The transaction is flagged as fraudulent. "
            "Write a professional response that acknowledges the fraud, reassures the customer, "
            "explains what actions will be taken (e.g., blocking the card immediately, launching an investigation, issuing a refund), "
            "and asks for any additional information if needed. "
            "{% else %}"
            "The transaction is legitimate. "
            "Write a professional response that confirms the transaction is valid, thanks the customer for reaching out, "
            "and offers additional support if they need anything else. "
            "{% endif %}"
            "The response should be in {{ language }}. "
            "Do not add any meta-commentary; only output the agent's response."
        ),
        model_alias=MODEL_ALIAS,
    )
)

data_designer.validate(config_builder)

[22:15:01] [INFO] ✅ Validation passed


### 🔁 Preview the dataset

Generate a small sample to verify quality and format.

日本語: 少数のサンプルを生成して品質を確認します。

In [9]:
preview = data_designer.preview(config_builder, num_records=3)

[22:15:01] [INFO] 🧐 Preview generation in progress
[22:15:01] [INFO]   |-- 🔒 Jinja rendering engine: secure
[22:15:01] [INFO] ✅ Validation passed
[22:15:01] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[22:15:01] [INFO] 🩺 Running health checks for models...
[22:15:01] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[22:15:01] [INFO]   |-- ✅ Passed!
[22:15:01] [INFO] ⚡ Using async task-queue preview
[22:15:01] [INFO] 📝 llm-text model config for column 'customer_message'
[22:15:01] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[22:15:01] [INFO]   |-- model alias: 'nemotron-nano-v3'
[22:15:01] [INFO]   |-- model provider: 'nvidia'
[22:15:01] [INFO]   |-- inference parameters:
[22:15:01] [INFO]   |  |-- generation_type=chat-completion
[22:15:01] [INFO]   |  |-- max_parallel_requests=4
[22:15:01] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable_thinking': False}}
[22:15:01] [INFO] 

In [10]:
# Display one record at a time
preview.display_sample_record()

                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name              ┃ Value                                                                                  ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ age               │ 48                                                                                     │
├───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ country           │ Australia                                                                              │
├───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ amount_usd        │ 2725                                                                                   │
├───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ merchant_category │ Entertainment                                                                          │
├───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ language          │ Japanese                                                                               │
├───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ scam_method       │ POS Purchase                                                                           │
├───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ is_fraud          │ 1                                                                                      │
├───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ customer_message  │ 不正なPOS購入で2725オーストラリアドルが引き落とされましたが、認可しておりません。ご確… │
├───────────────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ agent_response    │ お客様、                                                                               │
│                   │                                                                                        │
│                   │ ご報告いただき、誠にありがとうございます。                                             │
│                   │ ご利用いただいたエンターテインメント施設でのPOS購入金額が2,725円の取引が、Fraud        │
│                   │ Detectionシステムにより不正利用としてフラグされました。                                │
│                   │                                                                                        │
│                   │ まず、お客様の安全を最優先に考え、該当カードを即座に停止いたします。                   │
│                   │ 同時に、不正取引の詳細について徹底した調査を開始し、関連情報を精査いたします。         │
│                   │ 調査の結果、取引が誤認であることが判明した場合は、速やかにご指定の口座へ全額を返金い … │
│                   │                                                                                        │
│                   │ もし、取引時にご使用のカード番号、取引日時、利用された店舗名、またはその他ご不明点が … │
│                   │ お手数をお掛けしますが、追加でご提供いただける情報がございましたら、こちらのメールま … │
│                   │                                                                                        │
│                   │ ご不明点やご不安がございましたら、いつでもご連絡ください。                             │
│                   │ お客様の大切な資産を守るため、全力でサポートいたします。                               │
│                   │                                                                                        │
│                   │ 敬具                                                                                   │
│                   │ [銀行名] フraud Detection チーム                                                       │
└───────────────────┴────────────────────────────────────────────────────────────────────────────────────────┘

In [11]:
# Show as DataFrame
preview.dataset

,age,country,amount_usd,merchant_category,language,scam_method,is_fraud,customer_message,agent_response
0,48,Australia,2725,Entertainment,Japanese,POS Purchase,1,不正なPOS購入で2725オーストラリアドルが引き落とされましたが、認可しておりません。ご確...,お客様、\n\nご報告いただき、誠にありがとうございます。 \nご利用いただいたエンターテ...
1,71,South Africa,2647,Education,Japanese,SIM Swap,1,不正なSIMスワップ取引で$2647の支払いが行われましたが、私は一切承認していません。詐欺...,お客様、\n\nご報告いただき、誠にありがとうございます。ご指摘の2647ドルの取引が、SI...
2,27,Brazil,8421,Financial Services,Japanese,Romance Scam,1,不正なロマンス詐欺の取引（$8421）が unauthorized であり、詐欺防止の支援を...,"お客様\n\nご報告いただき、誠にありがとうございます。 \nご指摘いただいた8,421ド..."


### 📊 Analyze the generated data

Data Designer automatically generates basic statistics.

日本語: Data Designerは自動的に基本統計を生成します。

In [12]:
preview.analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 3                               │ 9                               │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                    ┃         data type ┃                number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ age                            │               int │                          3 (100.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ country                        │            string │                          3 (100.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ amount_usd                     │               int │                          3 (100.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ merchant_category              │            string │                          3 (100.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ language                       │            string │                           1 (33.3%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ scam_method                    │            string │                          3 (100.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ is_fraud                       │               int │                           1 (33.3%) │            bernoulli │
└────────────────────────────────┴───────────────────┴─────────────────────────────────────┴──────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                        ┃              ┃                            ┃     prompt tokens ┃      completion tokens ┃
┃ column name            ┃    data type ┃       number unique values ┃        per record ┃             per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━

### 🆙 Scale up!

Once satisfied, generate a larger dataset.

日本語: 満足したら、より大規模なデータセットを生成します。

In [13]:
results = data_designer.create(config_builder, num_records=10, dataset_name="fraud_conversations")
# TODO change 10 to 1000

[22:15:04] [INFO] 🎨 Creating Data Designer dataset
[22:15:04] [INFO]   |-- 🔒 Jinja rendering engine: secure
[22:15:04] [INFO] 📂 Dataset path '/workspace/asr/brev.nemo.curator.20260324/data_designer/artifacts/fraud_conversations' already exists. Dataset from this session
		     will be saved to '/workspace/asr/brev.nemo.curator.20260324/data_designer/artifacts/fraud_conversations_07-02-2026_221504' instead.
[22:15:04] [INFO] ✅ Validation passed
[22:15:04] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[22:15:04] [INFO] 🩺 Running health checks for models...
[22:15:04] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[22:15:04] [INFO]   |-- ✅ Passed!
[22:15:04] [INFO] ⚡ Using async task-queue builder
[22:15:04] [INFO] 📝 llm-text model config for column 'customer_message'
[22:15:04] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[22:15:04] [INFO]   |-- model alias: 'nemotron-nano-v3'
[22:15:04] [

In [14]:
dataset = results.load_dataset()
dataset.head()

,age,country,amount_usd,merchant_category,language,scam_method,is_fraud,customer_message,agent_response
0,47,Netherlands,3539,Utility,Japanese,Google Pay,0,この取引は私のものです。問題ないか確認してください。,"ご連絡いただき、誠にありがとうございます。 ご指摘の$3,539の取引について、当行のシ..."
1,59,Canada,2759,Retail Store,Japanese,ATM Cash Withdrawal,1,ATMで2759ドルの引き出しが unauthorized であり、詐欺の可能性があります。...,お客様 ご指摘の2759ドルの小売店での現金引き出し取引について、Fraud Alertが...
2,66,Nigeria,8274,Utility,Japanese,POS Purchase,1,"不正なPOS決済で8,274ドルが引き落とされました。詐欺であると確認し、速やかに調査・返金...",お客様、こんにちは。ご報告いただき、誠にありがとうございます。ご指摘の2023年11月2日1...
3,76,Australia,5917,Healthcare,Japanese,ATM Cash Withdrawal,0,ご連絡ありがとうございます。5917オーストラリアドルの医療関連のATM現金引き出し取引は私...,"ご連絡いただき、誠にありがとうございます。 ご指摘いただいた5,917ドルの取引について..."
4,68,South Africa,4600,Healthcare,Japanese,SIM Swap,1,この取引は偽造であり、私からの承認なしに行われました。詐欺行為を停止し、私の口座を保護するた...,"お客様 このたびはご連絡いただき、誠にありがとうございます。 ご報告いただいた4,60..."


In [15]:
analysis = results.load_analysis()
analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 10                              │ 9                               │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                    ┃         data type ┃                number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ age                            │               int │                           8 (80.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ country                        │            string │                           8 (80.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ amount_usd                     │               int │                         10 (100.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ merchant_category              │            string │                           4 (40.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ language                       │            string │                           1 (10.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ scam_method                    │            string │                           6 (60.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ is_fraud                       │               int │                           2 (20.0%) │            bernoulli │
└────────────────────────────────┴───────────────────┴─────────────────────────────────────┴──────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                        ┃              ┃                            ┃     prompt tokens ┃      completion tokens ┃
┃ column name            ┃    data type ┃       number unique values ┃        per record ┃             per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━

In [17]:
# save the dataset
print(type(dataset))
dataset.to_csv("fraud-detection-conversation-01.csv", index=False)

<class 'pandas.core.frame.DataFrame'>


## ⏭️ Next Steps

Now that you've generated a fraud detection conversation dataset, explore more advanced features:

- [Structured outputs and jinja expressions](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/2-structured-outputs-and-jinja-expressions/)
- [Seeding with an external dataset](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/)
- [Providing images as context](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)
- [Generating images](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)

日本語: 詐欺検出会話データセットを生成したので、さらに高度な機能を試してみましょう。